# 1. Problem Definition
   
1.1. Identify the Problem

The goal is to develop a machine learning model capable of distinguishing between text written by humans and text generated by AI models (e.g., ChatGPT, Gemini). This has applications in academic integrity, misinformation detection, content moderation, and authorship verification.


1.2. Define the Machine Learning Objective
This is a binary classification task:

Class 0: Human-written text

Class 1: AI-generated text

1.3. Specify the Output

The model should output either a class label (0 or 1) or a probability score indicating the likelihood that the input text is AI-generated.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
file_path = '/content/drive/MyDrive/Colab Notebooks/Final Project CLA /5000human_5000machine.csv'

df1 = pd.read_csv(file_path)

print(df1.head())

data = df1

   index                                               text  label  source  \
0     47  All the Love you do not see\n\nOn writing with...      0  medium   
1     85  Reading, writing and displaying images\n\nBefo...      0  medium   
2    255  Why I Think “Target Audience” Doesn’t Always M...      0  medium   
3    268  All sorts of books are published every year, a...      0  medium   
4    315  Finding The Right Question Not Answer\n\nWhen ...      0  medium   

   text_word_count label_cat  source_int  \
0              236     human           0   
1              386     human           0   
2              323     human           0   
3              176     human           0   
4              264     human           0   

                                      processed_text  
0  love see write heart hope photo joshua coleman...  
1  read write display image anything computer vis...  
2  think target audience always matter much first...  
3  sort book publish every year sort bookbuyer

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch
from sklearn.metrics import accuracy_score, classification_report
data = df1
texts = list(data['processed_text'])
labels = list(data['label'])
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

def tokenize_function(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
    encodings['labels'] = labels
    return encodings

train_encodings = tokenize_function(X_train, y_train)
test_encodings = tokenize_function(X_test, y_test)

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

train_dataset = CustomDataset(train_encodings)
test_dataset = CustomDataset(test_encodings)

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch', # ✅ تم التعديل من evaluation_strategy إلى eval_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
)

trainer.train()

results = trainer.evaluate()
print("Evaluation Results:")
print(results)

predictions = trainer.predict(test_dataset)
pred_labels = torch.argmax(torch.tensor(predictions.predictions), dim=1)

accuracy = accuracy_score(y_test, pred_labels.numpy())
report = classification_report(y_test, pred_labels.numpy(), target_names=['Human (0)', 'AI (1)'])

print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(report)

model.save_pretrained('./ai_vs_human_classifier_model')
tokenizer.save_pretrained('./ai_vs_human_classifier_tokenizer')

model = BertForSequenceClassification.from_pretrained('./ai_vs_human_classifier_model')
tokenizer = BertTokenizer.from_pretrained('./ai_vs_human_classifier_tokenizer')

def predict(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=-1)
    return predictions

texts = ["This is a simple email I wrote quickly.",
         "It is imperative that the discourse surrounding the existential threats posed by climate change is immediately elevated to a national security priority."]
predictions = predict(texts)
print("Predictions:", predictions)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3257771105.py:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: You chose "Don't visualize my results"


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
